# Stage 4: Real-World Distribution Comparison

## Goal

This notebook compares CLIP demographic-occupation association gaps from the main FairFace notebook with aggregate labour-statistics distributions where a defensible occupation mapping exists. The purpose is to support the hypothesis that CLIP associations may diverge from or amplify real-world occupation distributions, while avoiding claims that FairFace images have true occupation labels.


## Inputs

Required outputs from the main notebook:

- `outputs/fairface_facet_broad/tables/gender_gap_by_occupation.csv`
- `outputs/fairface_facet_broad/tables/race_gap_by_occupation.csv` if race comparison is later extended

Required config/data for the feasible BLS comparison:

- `configs/occupation_realworld_mapping.csv`
- `data/bls/raw/cpsaat11.xlsx` or `bls/raw/cpsaat11.xlsx`

ABS data is checked separately. If the available ABS file cannot support the intended comparison, this notebook writes a diagnostic note instead of fabricating results.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clip_bias.data import resolve_dataset_path
from clip_bias.realworld import (
    build_bls_gender_comparison,
    load_bls_gender_distribution,
    load_occupation_mapping,
    plot_clip_vs_bls_gender_gap,
    plot_realworld_gap_rank_difference,
    write_todo_note,
)

try:
    display
except NameError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

MAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "fairface_facet_broad"
MAIN_TABLE_DIR = MAIN_OUTPUT_DIR / "tables"
REALWORLD_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "realworld"
REALWORLD_TABLE_DIR = REALWORLD_OUTPUT_DIR / "tables"
REALWORLD_FIGURE_DIR = REALWORLD_OUTPUT_DIR / "figures"
REALWORLD_TABLE_DIR.mkdir(parents=True, exist_ok=True)
REALWORLD_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = PROJECT_ROOT / "configs" / "occupation_realworld_mapping.csv"
GENDER_GAP_PATH = MAIN_TABLE_DIR / "gender_gap_by_occupation.csv"
RACE_GAP_PATH = MAIN_TABLE_DIR / "race_gap_by_occupation.csv"

print("Project root:", PROJECT_ROOT)
print("Main output tables:", MAIN_TABLE_DIR)
print("Real-world output:", REALWORLD_OUTPUT_DIR)


## Method

The comparison joins occupation-level CLIP gender association gaps to BLS percent-women occupation statistics using a manual mapping table. The mapping is deliberately conservative: only rows with both a CLIP occupation and a BLS occupation statistic are included in the final comparison.

The CLIP metric is `male_minus_female`, the difference between mean CLIP image-text similarity for male-labelled and female-labelled FairFace images. The BLS metric is converted to `male % - female %` from the BLS percent-women column. These quantities are not on the same measurement scale, so the comparison focuses on direction, rank, and qualitative divergence rather than direct numeric equality.


In [ ]:
if not GENDER_GAP_PATH.exists():
    raise FileNotFoundError(
        f"Missing {GENDER_GAP_PATH}. Run src/notebooks/04_fairface_facet_broad_occupation_bias.ipynb first."
    )

gender_gap = pd.read_csv(GENDER_GAP_PATH)
race_gap = pd.read_csv(RACE_GAP_PATH) if RACE_GAP_PATH.exists() else None

mapping = load_occupation_mapping(
    MAPPING_PATH,
    clip_occupations=gender_gap["occupation"].dropna().astype(str).tolist(),
)

print("CLIP gender-gap rows:", gender_gap.shape)
print("Mapping rows:", mapping.shape)
print("Mapped occupations present in CLIP outputs:", int(mapping.get("in_clip_outputs", True).sum()))
print("Unmapped CLIP occupations:", len(mapping.attrs.get("unmapped_clip_occupations", [])))
display(mapping.head(30))


preliminary_mapping_path = REALWORLD_TABLE_DIR / "occupation_realworld_mapping_used.csv"
mapping.to_csv(preliminary_mapping_path, index=False)
print("Saved preliminary mapping diagnostic:", preliminary_mapping_path)


## Occupation Mapping Limitations

Occupation mapping is a major source of uncertainty. Some FACET-derived prompt labels are roles, activities, or colloquial terms rather than formal labour-statistics categories. The notebook therefore saves `occupation_realworld_mapping_used.csv`, including flags for whether each mapped row has a CLIP gap and a BLS statistic. Only rows marked usable should be discussed in the report.


## BLS Comparison


In [ ]:
bls_comparison_path = REALWORLD_TABLE_DIR / "clip_vs_bls_gender_gap.csv"
mapping_used_path = REALWORLD_TABLE_DIR / "occupation_realworld_mapping_used.csv"
bls_scatter_path = REALWORLD_FIGURE_DIR / "clip_vs_bls_gender_gap_scatter.png"
rank_difference_path = REALWORLD_FIGURE_DIR / "realworld_amplification_top_occupations.png"
bls_todo_path = REALWORLD_TABLE_DIR / "bls_comparison_todo.md"

try:
    bls_root = resolve_dataset_path(PROJECT_ROOT, "bls")
    bls_path = bls_root / "raw" / "cpsaat11.xlsx"
    bls_gender = load_bls_gender_distribution(bls_path)
    clip_vs_bls, mapping_used = build_bls_gender_comparison(gender_gap, mapping, bls_gender)

    mapping_used.to_csv(mapping_used_path, index=False)

    if clip_vs_bls.empty:
        raise ValueError("No mapped occupations had both CLIP gender gaps and BLS percent-women values.")

    clip_vs_bls.to_csv(bls_comparison_path, index=False)
    plot_clip_vs_bls_gender_gap(clip_vs_bls, bls_scatter_path)
    plot_realworld_gap_rank_difference(clip_vs_bls, rank_difference_path, top_n=20)

    print("Saved:", bls_comparison_path)
    print("Saved:", mapping_used_path)
    display(clip_vs_bls.head(30))

except Exception as exc:
    write_todo_note(
        bls_todo_path,
        title="BLS comparison TODO",
        reason=str(exc),
        next_steps=[
            "Run the main CLIP association notebook first so gender_gap_by_occupation.csv exists.",
            "Place cpsaat11.xlsx under data/bls/raw/ or bls/raw/.",
            "Review configs/occupation_realworld_mapping.csv and keep only defensible occupation mappings.",
        ],
    )
    print(f"BLS comparison not completed. Wrote diagnostic note to {bls_todo_path}")


## ABS Comparison If Feasible

The available ABS workbook is useful labour-statistics context, but it is not automatically comparable to FairFace race labels. If it does not contain a directly usable race/ethnicity distribution aligned with the CLIP demographic analysis and a reviewed occupation mapping, this notebook records that limitation and does not fabricate an ABS comparison.


In [ ]:
abs_todo_path = REALWORLD_TABLE_DIR / "abs_comparison_todo.md"
abs_comparison_path = REALWORLD_TABLE_DIR / "clip_vs_abs_comparison.csv"

try:
    abs_root = resolve_dataset_path(PROJECT_ROOT, "abs")
    abs_raw_dir = abs_root / "raw"
    abs_files = sorted(abs_raw_dir.glob("*.xlsx"))
    if not abs_files:
        raise FileNotFoundError(f"No ABS workbooks found under {abs_raw_dir}")

    reason = (
        "ABS workbook(s) were found, but this project does not yet include a reviewed "
        "FACET/FairFace occupation-to-ANZSCO mapping or an ABS race/ethnicity table aligned "
        "with the FairFace demographic categories. Reporting an ABS race comparison would "
        "therefore overclaim the available evidence."
    )
    write_todo_note(
        abs_todo_path,
        title="ABS comparison TODO",
        reason=reason + " Files found: " + ", ".join(path.name for path in abs_files),
        next_steps=[
            "Create a reviewed occupation-to-ANZSCO mapping table.",
            "Identify an ABS table with demographic variables that match the intended comparison.",
            "Generate clip_vs_abs_comparison.csv only after the mapping and demographic variables are defensible.",
        ],
    )
    print(f"ABS comparison not generated. Wrote limitation note to {abs_todo_path}")

except Exception as exc:
    write_todo_note(
        abs_todo_path,
        title="ABS comparison TODO",
        reason=str(exc),
        next_steps=[
            "Place ABS workbooks under data/abs/raw/ or abs/raw/ if an ABS comparison is needed.",
            "Add a reviewed ANZSCO mapping before reporting ABS results.",
        ],
    )
    print(f"ABS comparison not completed. Wrote diagnostic note to {abs_todo_path}")


## Interpretation

Use the generated `clip_vs_bls_gender_gap.csv` only if it exists. Interpret it as a mapped aggregate comparison, not as individual-level ground truth. Agreement in direction may suggest that CLIP reflects broad labour-market stereotypes or distributions; disagreement may indicate CLIP-specific associations, prompt effects, mapping noise, or dataset effects. Because CLIP similarity and labour-force percentages are different quantities, avoid saying one numerically equals or directly measures the other.


In [ ]:
if bls_comparison_path.exists():
    clip_vs_bls = pd.read_csv(bls_comparison_path)
    print("Mapped BLS comparison rows:", len(clip_vs_bls))
    display(clip_vs_bls.sort_values("abs_gender_gap", ascending=False).head(10))
else:
    print("No BLS comparison CSV found. Use the TODO note and do not report BLS findings yet.")


## Report-ready Notes

After running this notebook, report only mapped occupations from `clip_vs_bls_gender_gap.csv`. A careful sentence template is:

"For the subset of FACET-derived occupations with defensible BLS mappings, we compared CLIP male-minus-female association gaps with BLS aggregate occupation gender distributions. The comparison is descriptive and aggregate-level; it does not imply that FairFace images have true occupation labels."

If the CSV has not been generated, write: "The real-world comparison remains a planned analysis because the required labour-statistics mapping/data were incomplete."


## Limitations and Ethical Caution

FairFace images do not contain true occupation labels. CLIP similarity is not a claim about a person?s real occupation, ability, or preference. BLS and ABS are aggregate labour statistics and cannot be directly matched to individual images. Only mapped occupations should be discussed, and mapping uncertainty should be acknowledged in the paper.
